# Exploring the SMS Spam Collection

Before training any model, we look at the data. This step is called **EDA**
(exploratory data analysis). The goal is to answer simple questions:

1. What do the messages look like?
2. How many are spam and how many are ham (normal messages)?
3. Are spam messages longer or shorter than ham?
4. Which words show up most in each class?
5. Are there duplicate messages?

Run each cell with **Shift+Enter**.

## Setup

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Let the notebook import our own code from the src/ folder.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from data import load_data

# One colour per class, used in every chart below.
COLORS = {"ham": "#2a78d6", "spam": "#eb6834"}

# Show the full message text instead of cutting it off with "..."
pd.set_option("display.max_colwidth", None)

# A cleaner default look for charts.
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e1e0d9",
})

## 1. Load the data

`load_data()` is the function we wrote in `src/data.py`. `shape` gives
(rows, columns).

In [ ]:
messages = load_data()
messages.shape

In [ ]:
messages.head()

## 2. Read some messages by eye

Numbers come later. First, just read. `sample(10, random_state=42)` picks 10
random rows; the fixed `random_state` means we get the same 10 every time.

In [ ]:
messages[messages["label"] == "ham"].sample(10, random_state=42)

In [ ]:
messages[messages["label"] == "spam"].sample(10, random_state=42)

## 3. How many spam vs ham?

`value_counts()` counts how often each value appears in a column.

In [ ]:
counts = messages["label"].value_counts()
percents = (messages["label"].value_counts(normalize=True) * 100).round(1)

pd.DataFrame({"count": counts, "percent": percents})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    counts.index,
    counts.values,
    color=[COLORS[label] for label in counts.index],
    width=0.5,
)
bar_labels = [f"{count:,} ({percent}%)" for count, percent in zip(counts, percents)]
ax.bar_label(bars, labels=bar_labels, padding=3)

ax.set_title("Messages per class")
ax.set_ylabel("Number of messages")
ax.grid(axis="x", visible=False)
plt.show()

The classes are very uneven. This is called **class imbalance**, and it matters
later: a "model" that always answers *ham* would be right about 87% of the time
while catching no spam at all. So accuracy alone will not tell us if our model
is any good.

## 4. Message length

We add a new column with the number of characters in each message, then
compare the two classes. `describe()` gives summary statistics: `mean` is the
average, `50%` is the median (the middle value).

In [ ]:
messages["length"] = messages["message"].str.len()

messages.groupby("label")["length"].describe().round(1)

In [ ]:
MAX_LENGTH_SHOWN = 300
bins = range(0, MAX_LENGTH_SHOWN + 10, 10)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

for ax, label in zip(axes, ["ham", "spam"]):
    lengths = messages.loc[messages["label"] == label, "length"]
    ax.hist(lengths, bins=bins, color=COLORS[label], edgecolor="white")
    ax.set_title(f"{label} message length", loc="left")
    ax.set_ylabel("Messages")

axes[-1].set_xlabel("Characters in message")
fig.tight_layout()
plt.show()

longer = (messages["length"] > MAX_LENGTH_SHOWN).sum()
print(f"{longer} messages are longer than {MAX_LENGTH_SHOWN} characters and are not shown.")

Each chart is a **histogram**: every bar counts the messages whose length falls
in a 10-character range. The two charts share the same x-axis so the shapes can
be compared directly (their y-axes differ because there is far more ham).

## 5. Most common words

We split each message into lowercase words and count them. We skip **stop
words**: very common words such as "the", "and", "is" that appear everywhere
and tell us little.

In [ ]:
def count_words(texts):
    """Count how often each word appears, ignoring stop words."""
    counter = Counter()
    for text in texts:
        words = re.findall(r"[a-z']+", text.lower())
        counter.update(
            word for word in words
            if word not in ENGLISH_STOP_WORDS and len(word) > 1
        )
    return counter


ham_words = count_words(messages.loc[messages["label"] == "ham", "message"])
spam_words = count_words(messages.loc[messages["label"] == "spam", "message"])

TOP_N = 15
top_ham = pd.DataFrame(ham_words.most_common(TOP_N), columns=["word", "count"])
top_spam = pd.DataFrame(spam_words.most_common(TOP_N), columns=["word", "count"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, label, top in zip(axes, ["ham", "spam"], [top_ham, top_spam]):
    ax.barh(top["word"], top["count"], color=COLORS[label])
    ax.invert_yaxis()  # most common word at the top
    ax.set_title(f"Top {TOP_N} words in {label}", loc="left")
    ax.set_xlabel("Times the word appears")
    ax.grid(axis="y", visible=False)

fig.tight_layout()
plt.show()

Two odd "words" in the ham chart, `lt` and `gt`, are not real words. The raw
file contains leftover web codes `&lt;` and `&gt;` (the HTML way of writing
`<` and `>`), and our simple word splitter keeps the letters. It is a reminder
that real data is messy, and worth reading before trusting.

## 6. Duplicate messages

`duplicated()` marks a row as `True` if the same message text already appeared
in an earlier row.

In [ ]:
is_duplicate = messages.duplicated(subset="message")

print(f"Duplicate rows: {is_duplicate.sum()} of {len(messages)}")
print(f"Unique messages: {messages['message'].nunique()}")
print()
print("Duplicate rows per class:")
print(messages.loc[is_duplicate, "label"].value_counts())

In [ ]:
# The messages that repeat the most.
messages["message"].value_counts().head(5)

In [ ]:
# Does any message text appear with both labels? (That would be a labelling error.)
labels_per_message = messages.groupby("message")["label"].nunique()
print(f"Messages labelled both ham and spam: {(labels_per_message > 1).sum()}")

**Decision: drop the duplicates before training.** If the same message sits in
both the training data and the test data, the model is tested on something it
has already seen, and its score looks better than it really is. We will do the
dropping in the next phase, just before splitting the data.

## Findings

- **The classes are imbalanced.** 4,827 messages are ham (86.6%) and 747 are spam (13.4%). Accuracy alone will mislead us, so we will also measure precision and recall.
- **Spam is longer and more uniform in length.** The median spam message is 149 characters against 52 for ham, and most spam sits just under the 160-character limit of a single SMS. Ham varies widely, from 2 to 910 characters.
- **The two classes use different words.** Spam is full of "free", "txt", "claim", "prize", "cash" and "won"; ham is everyday chat such as "ok", "know", "good", "come" and "love". This is the signal a word-counting model can learn from.
- **Some words appear in both classes** ("ur", "just"), so no single word decides the answer; the model has to weigh many words together.
- **403 of the 5,574 rows are duplicates** (309 ham, 94 spam), leaving 5,171 unique messages. No message carries both labels. We will drop the duplicates before splitting into training and test data.
- **The text is messy.** It has SMS slang ("u", "ur", "gr8"), leftover web codes (`&lt;`, `&gt;`), and many phone numbers and prices in spam. Our word counts above ignored digits, so numbers are something to look at when we turn text into features.